# Vi-Graph — final evaluation (Phase 7, last GPU run)

The last GPU run of the project. It evaluates the trained adapter (`RUN_NAME`) and the
zero-shot model on the **whole held-out test split (500 diagrams)** with the settings chosen in
Phase 7 (896 px, 2048 tokens, runaway guard on), plus the OCR baseline:

| Section | What it does | Time (RTX 4080 SUPER, rough) |
| --- | --- | --- |
| 1. Setup | Settings, helpers, system check, latest code, libraries, GPU test (all quick when already installed) | ~2 min |
| 2. Dataset | Checks the dataset the main notebook built (rebuilds it only if missing) | ~1 min |
| A. Greedy, all 500 | Fine-tuned, zero-shot, OCR baseline; paired comparisons; hand-back zip | ~12–14 h |
| B. 3 sampling seeds per model | Run-to-run variance (§20.10): 3 seeds × 2 models; seed statistics; hand-back zip | ~35–45 h (all 500) or ~9 h (`SEED_LIMIT = 112`) |

**How to run it:** put this notebook in `E:\Asif\vigraph` next to the main notebook, open it,
**Select Kernel → `.venv`** (the same one), and run the cells in order. Every run is
resumable: after a stop or restart, run sections 1–2 and A.1 again, then the cell you were in.
Keep sleep set to *Never*.

**Send section A's zip as soon as A is done** (cell A.6), before starting B, so the write-up can
start while B runs. All runs must stay on this PC's GPU (the runner refuses to resume a run on a
different one).

## 1. Setup

### 1.1 Settings (edit here if needed)

In [ ]:
# ---- Settings (the same folder, run and adapter as the main notebook) --------------------
from pathlib import Path

ROOT = Path(r"E:\Asif\vigraph")   # the main notebook's folder: repo, data, model cache, runs, results
REPO_URL = "https://github.com/asifuddin01/Vi-Graph.git"
BRANCH = "claude/tender-ramanujan-70dwgj"

VRAM_LIMIT_GB = 10.5             # PyTorch limit, as in the main notebook
RUN_NAME = "qwen3vl-2b-qlora-a6000-v1"   # the trained adapter: output\runs\<RUN_NAME>\adapter
COMPUTE_DTYPE = "bfloat16"
RUN_BASELINE = True              # OCR baseline on all 500 (CPU) — skipped if Tesseract is missing
DATA_WORKERS = 8                 # only used if the dataset has to be rebuilt

# ---- Final evaluation --------------------------------------------------------------------
IMAGE_MAX_SIDE = 896             # the adapter's training size; 768–1024 scored the same
MAX_NEW_TOKENS = 2048            # 4096 changed nothing
RUNAWAY_GUARD = True             # same scores, 26 % less time
SEEDS = [0, 1, 2]                # section B: one sampled run per seed and model
TEMPERATURE = 0.7                # section B sampling (greedy has no run-to-run variance)
TOP_P = 0.8
SEED_LIMIT = None                # section B samples per run: None = all 500 (~35–45 h),
                                 # 112 = the stratified subset used so far (~9 h)


# Portable Graphviz (no install/admin), tried in order if Graphviz is not already installed
GRAPHVIZ_VERSIONS = ["16.1.0", "16.0.0", "15.1.1"]
GRAPHVIZ_URL = ("https://gitlab.com/api/v4/projects/4207231/packages/generic/graphviz-releases/"
                "{v}/windows_10_cmake_Release_Graphviz-{v}-win64.zip")

# Library versions: the exact set that trained and evaluated successfully in Colab
TORCH_INDEX = "https://download.pytorch.org/whl/cu128"
PINS = {"torch": "2.11.0", "torchvision": "0.26.0", "transformers": "5.17.0",
        "peft": "0.21.1", "bitsandbytes": "0.50.2", "accelerate": "1.15.0"}

### 1.2 Paths, environment and helpers

Writes the small driver script (`E:\Asif\vigraph\tools\vigraph_windows.py`) that every GPU step
runs through: it applies the VRAM cap, makes file writing identical to Linux (otherwise the
dataset's hash checks fail on Windows), and keeps data loading in one process.

In [ ]:
import codecs, json, os, shutil, subprocess, sys, time

if os.name != "nt":
    raise RuntimeError("This notebook is for Windows. Use the Colab notebook elsewhere.")
if sys.prefix == sys.base_prefix:
    raise RuntimeError("The kernel is not the venv: in VS Code click 'Select Kernel' (top right) "
                       "→ Python Environments → .venv, then run this cell again.")
if not (3, 11) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError(f"The venv uses Python {sys.version.split()[0]}; it needs 3.11, 3.12 or 3.13.")

PY = Path(sys.executable)
REPO = ROOT / "Vi-Graph"
DATA_DIR = ROOT / "data" / "synthetic-v1"
OUT = ROOT / "output"
RUN_DIR = OUT / "runs" / RUN_NAME
SMOKE_DIR = OUT / "runs" / "_smoke"
EVAL_DIR = OUT / "eval"
LOG_DIR = OUT / "logs"
PROBE_FILE = OUT / "memory_probe.json"
TOOLS = ROOT / "tools"
DRIVER = TOOLS / "vigraph_windows.py"
CONFIG = REPO / "training" / "configs" / "qlora_t4.yaml"   # base config; Windows overrides below
HF_HOME = ROOT / "hf-cache"
for folder in (OUT, LOG_DIR, TOOLS, HF_HOME, ROOT / "data"):
    folder.mkdir(parents=True, exist_ok=True)

# Tools may be installed per-user, system-wide, or downloaded portably into E:\Asif\vigraph\tools.
def add_to_path(folder):
    folder = str(folder)
    if Path(folder).is_dir() and folder.lower() not in os.environ["PATH"].lower():
        os.environ["PATH"] = folder + os.pathsep + os.environ["PATH"]
    if "ENV" in globals():
        ENV["PATH"] = os.environ["PATH"]


local = os.path.expandvars(r"%LOCALAPPDATA%\Programs")
for folder in (r"C:\Program Files\Git\cmd", local + r"\Git\cmd",
               r"C:\Program Files\Graphviz\bin", local + r"\Graphviz\bin",
               r"C:\Program Files\Tesseract-OCR", r"C:\Program Files (x86)\Tesseract-OCR",
               local + r"\Tesseract-OCR", TOOLS / "tesseract",
               *sorted(TOOLS.glob("graphviz/*/bin"))):
    add_to_path(folder)

ENV = {**os.environ,
       "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1",
       "PYTHONPATH": os.pathsep.join([str(REPO / "backend"), str(REPO)]),
       "VIGRAPH_REPO": str(REPO), "VIGRAPH_VRAM_GB": str(VRAM_LIMIT_GB),
       "HF_HOME": str(HF_HOME), "HF_HUB_DISABLE_SYMLINKS_WARNING": "1",
       "TOKENIZERS_PARALLELISM": "false", "TQDM_MININTERVAL": "5",
       "HF_HUB_DOWNLOAD_TIMEOUT": "60",
       # temp files on F: (not C:\Users\...\Temp, where antivirus locked pip's download)
       "TMP": str(ROOT / "tmp"), "TEMP": str(ROOT / "tmp")}
(ROOT / "tmp").mkdir(exist_ok=True)


def run(cmd, log=None, check=True):
    """Run a command, stream its output live here, and append it to output/logs/<log>.log."""
    cmd = [str(c) for c in cmd]
    line = subprocess.list2cmdline(cmd)
    print("$", line, flush=True)
    log_path = LOG_DIR / f"{log}.log" if log else None
    handle = open(log_path, "a", encoding="utf-8") if log_path else None
    if handle:
        handle.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')}  $ {line}\n")
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    proc = subprocess.Popen(cmd, cwd=str(REPO if REPO.exists() else ROOT), env=ENV,
                            stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT)
    try:
        while chunk := proc.stdout.read1(65536):
            text = decoder.decode(chunk)
            sys.stdout.write(text)
            sys.stdout.flush()
            if handle:
                handle.write(text)
                handle.flush()
        code = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("\n[stopped — re-run this cell to continue where it left off]")
        raise
    finally:
        if handle:
            handle.close()
    if check and code != 0:
        where = f"full output: {log_path}" if log_path else "see the output above"
        raise RuntimeError(f"Command failed with exit code {code} ({where}).")
    return code


def driver(*args, log=None):
    return run([PY, DRIVER, *args], log=log)


def probe_result():
    return json.loads(PROBE_FILE.read_text(encoding="utf-8")) if PROBE_FILE.exists() else None


def image_side():
    """The image size for training and evaluation: the memory probe's choice (section 4)."""
    probe = probe_result()
    if probe is None:
        return 1024  # the config default, until section 4 has measured
    if probe.get("chosen_image_max_side") is None:
        raise RuntimeError(
            f"The memory probe found no image size that fits VRAM_LIMIT_GB = {probe['cap_gb']:.1f}. "
            "Raise VRAM_LIMIT_GB in Settings (1.1), re-run section 1, then section 4.")
    return probe["chosen_image_max_side"]


def overrides(with_image_side=True):
    """Changes on top of qlora_t4.yaml, identical for every step (resume checks depend on it).
    The preprocessing check (section 3) runs before the probe, so it uses the config default."""
    values = {"run_name": RUN_NAME, "quantization.compute_dtype": COMPUTE_DTYPE}
    if with_image_side:
        values["data.image_max_side"] = image_side()
    return json.dumps(values)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def download(url, dest):
    """Download with a progress line; returns dest. Uses Windows' proxy settings."""
    import urllib.request
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    partial = dest.with_suffix(dest.suffix + ".part")
    request = urllib.request.Request(url, headers={"User-Agent": "vigraph-notebook"})
    with urllib.request.urlopen(request, timeout=60) as response, open(partial, "wb") as out:
        total = int(response.headers.get("Content-Length") or 0)
        done = 0
        while chunk := response.read(1 << 20):
            out.write(chunk)
            done += len(chunk)
            size = f" / {total / 1024**2:.0f} MB" if total else ""
            print(f"\r  downloading {dest.name}: {done / 1024**2:.0f} MB{size}", end="", flush=True)
    print()
    partial.replace(dest)
    return dest


def remove_tree(path):
    """shutil.rmtree that also removes read-only files (e.g. old .git objects on Windows)."""
    import stat

    def make_writable(func, target, _):
        os.chmod(target, stat.S_IWRITE)
        func(target)
    if Path(path).exists():
        if sys.version_info >= (3, 12):
            shutil.rmtree(path, onexc=make_writable)
        else:
            shutil.rmtree(path, onerror=make_writable)


DRIVER_SOURCE = r'''"""Vi-Graph Windows driver — written by vigraph_qlora_windows.ipynb; do not edit by hand.

Every GPU step of the notebook runs through this script in its own process, so that:
  * PyTorch is capped at VIGRAPH_VRAM_GB of GPU memory (the rest of the card stays free);
  * files are written exactly as on Linux/Colab (UTF-8, "\\n" line endings). Windows would
    otherwise write ground-truth JSON with "\\r\\n", and the dataset's sha256 checks would fail;
  * DataLoader worker processes (which Windows starts with "spawn") are not used.

Usage (from the repo root, with the venv's python):
    python vigraph_windows.py gpu
    python vigraph_windows.py check  --data DIR --config YAML --overrides JSON
    python vigraph_windows.py probe  --data DIR --config YAML --overrides JSON --out FILE
    python vigraph_windows.py smoke  --data DIR --config YAML --overrides JSON --out DIR
    python vigraph_windows.py train  --data DIR --config YAML --overrides JSON --out DIR
    python vigraph_windows.py handback --out ZIP --train DIR --eval DIR... --extra FILE...
    python vigraph_windows.py module <python module> [its arguments...]
"""

from __future__ import annotations

import argparse
import gc
import json
import os
import pathlib
import runpy
import shutil
import sys
import time
from pathlib import Path

REPO = Path(os.environ.get("VIGRAPH_REPO", Path.cwd())).resolve()
for entry in (str(REPO / "backend"), str(REPO)):
    if entry not in sys.path:
        sys.path.insert(0, entry)


# --- Linux-identical text files ------------------------------------------------------
_open = pathlib.Path.open
_write_text = pathlib.Path.write_text
_read_text = pathlib.Path.read_text


def _posix_open(self, mode="r", buffering=-1, encoding=None, errors=None, newline=None):
    if "b" not in mode:
        encoding = encoding or "utf-8"
        if newline is None and any(flag in mode for flag in "wax+"):
            newline = "\n"  # no "\r\n" translation on write
    return _open(self, mode, buffering, encoding, errors, newline)


def _posix_write_text(self, data, encoding=None, errors=None, newline=None):
    return _write_text(
        self, data, encoding=encoding or "utf-8", errors=errors,
        newline="\n" if newline is None else newline,
    )


def _utf8_read_text(self, encoding=None, errors=None, *args, **kwargs):
    return _read_text(self, encoding or "utf-8", errors, *args, **kwargs)


pathlib.Path.open = _posix_open
pathlib.Path.write_text = _posix_write_text
pathlib.Path.read_text = _utf8_read_text


# --- GPU memory cap ------------------------------------------------------------------
def cap_vram(require_gpu: bool = True) -> float | None:
    """Cap PyTorch's allocator at VIGRAPH_VRAM_GB; returns the cap in bytes."""
    import torch

    if not torch.cuda.is_available():
        if require_gpu:
            raise SystemExit(
                "CUDA is not available to PyTorch. Re-run the 'Install libraries' cell: a CPU-only "
                "torch was installed, or the NVIDIA driver is too old for CUDA 12.8."
            )
        return None
    limit_gb = float(os.environ.get("VIGRAPH_VRAM_GB") or 0)
    total = torch.cuda.get_device_properties(0).total_memory
    if limit_gb <= 0:
        return float(total)
    fraction = min(1.0, limit_gb * 1024**3 / total)
    torch.cuda.set_per_process_memory_fraction(fraction, 0)
    print(
        f"[vram] PyTorch limited to {limit_gb:.1f} GB of {total / 1024**3:.1f} GB "
        f"on {torch.cuda.get_device_name(0)}",
        flush=True,
    )
    return fraction * total


def patch_trainer_for_windows() -> None:
    """Load training data in the main process (no spawn-started DataLoader workers)."""
    import transformers

    base = transformers.TrainingArguments
    if getattr(base, "_vigraph_windows", False):
        return
    global WindowsTrainingArguments

    class WindowsTrainingArguments(base):  # noqa: D401 — picklable (module level name below)
        _vigraph_windows = True

        def __post_init__(self):
            self.dataloader_num_workers = 0
            super().__post_init__()

    WindowsTrainingArguments.__qualname__ = "WindowsTrainingArguments"
    WindowsTrainingArguments.__module__ = __name__
    transformers.TrainingArguments = WindowsTrainingArguments


def load_train_config(args: argparse.Namespace, **extra: object):
    from training.qlora.config import load_config

    overrides = json.loads(args.overrides or "{}")
    overrides.update(extra)
    return load_config(Path(args.config), **overrides)


# --- commands --------------------------------------------------------------------------
def cmd_gpu(args: argparse.Namespace) -> int:
    import torch

    print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| python", sys.version.split()[0])
    cap = cap_vram()
    name = torch.cuda.get_device_name(0)
    print("GPU:", name, "| bf16 supported:", torch.cuda.is_bf16_supported())
    import bitsandbytes as bnb

    layer = bnb.nn.Linear4bit(
        256, 256, compute_dtype=torch.bfloat16, quant_type="nf4"
    ).cuda()
    out = layer(torch.randn(4, 256, device="cuda", dtype=torch.bfloat16))
    assert out.shape == (4, 256) and torch.isfinite(out).all()
    weight = torch.nn.Parameter(torch.randn(128, 128, device="cuda"))
    optimizer = bnb.optim.PagedAdamW8bit([weight], lr=1e-3)
    weight.sum().backward()
    optimizer.step()
    print("bitsandbytes", bnb.__version__, "4-bit layer + paged 8-bit AdamW: OK")
    import accelerate
    import peft
    import transformers

    print(
        f"transformers {transformers.__version__} | peft {peft.__version__} | "
        f"accelerate {accelerate.__version__}"
    )
    print(f"memory cap: {cap / 1024**3:.1f} GB — all checks passed")
    return 0


def cmd_check(args: argparse.Namespace) -> int:
    import statistics

    from transformers import AutoProcessor

    from training.qlora.collate import Collator
    from training.qlora.data import build_example, load_training_split

    config = load_train_config(args)
    data = Path(args.data)
    train_info, train = load_training_split(data, config.data.train_split)
    val_info, val = load_training_split(data, config.data.eval_split)
    print(
        f"train {len(train)} (hash {train_info.split_hash[:12]}), "
        f"val {len(val)} (hash {val_info.split_hash[:12]})"
    )
    processor = AutoProcessor.from_pretrained(config.model.id, revision=config.model.revision)
    collator = Collator(processor)
    trained = collator.trained_text(build_example(train[9], config.data.image_max_side))
    print("Trained on:", trained[:160], "…", trained[-40:])
    if not trained.startswith('{"schema_version":"2.0"'):
        raise SystemExit("label masking looks wrong — stop here and send this output")
    lengths, answers = [], []
    for sample in train[::50]:
        batch = collator([build_example(sample, config.data.image_max_side)])
        lengths.append(batch["input_ids"].shape[1])
        answers.append(int((batch["labels"] != -100).sum()))
    print(
        f"sequence tokens: mean {statistics.mean(lengths):.0f}, max {max(lengths)}; "
        f"answer tokens: mean {statistics.mean(answers):.0f}, max {max(answers)}"
    )
    print("preprocessing check passed")
    return 0


def _load_qlora_model(config):
    import torch
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import AutoModelForImageTextToText, BitsAndBytesConfig

    dtype = getattr(torch, config.quantization.compute_dtype)
    model = AutoModelForImageTextToText.from_pretrained(
        config.model.id,
        revision=config.model.revision,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=config.quantization.load_in_4bit,
            bnb_4bit_quant_type=config.quantization.quant_type,
            bnb_4bit_use_double_quant=config.quantization.double_quant,
            bnb_4bit_compute_dtype=dtype,
        ),
        dtype=dtype,
        device_map={"": 0},
    )
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
    for cfg in (model.config, getattr(model.config, "text_config", None)):
        if cfg is not None:
            cfg.use_cache = False
    lora = config.lora
    model = get_peft_model(
        model,
        LoraConfig(
            r=lora.r, lora_alpha=lora.alpha, lora_dropout=lora.dropout,
            target_modules=lora.target_modules, bias="none", task_type="CAUSAL_LM",
        ),
    )
    return model, dtype


def cmd_probe(args: argparse.Namespace) -> int:
    """Forward + backward on the longest training samples under the memory cap, trying
    image_max_side values from largest to smallest; picks the first that fits with headroom."""
    import torch
    from transformers import AutoProcessor

    from training.qlora.collate import Collator
    from training.qlora.data import build_example, load_training_split

    cap = cap_vram()
    sizes = [int(s) for s in args.sizes.split(",")]
    config = load_train_config(args, **{"data.image_max_side": sizes[0]})
    _, samples = load_training_split(Path(args.data), config.data.train_split)
    processor = AutoProcessor.from_pretrained(config.model.id, revision=config.model.revision)
    collator = Collator(processor)

    print(f"measuring {len(samples)} training sequences at image_max_side={sizes[0]} …", flush=True)
    lengths = []
    for index, sample in enumerate(samples):
        batch = collator([build_example(sample, sizes[0])])
        lengths.append((int(batch["input_ids"].shape[1]), index))
        if (index + 1) % 250 == 0:
            print(f"  {index + 1}/{len(samples)}", flush=True)
    lengths.sort(reverse=True)
    longest = [samples[i] for _, i in lengths[: args.top]]
    print(f"longest sequences: {[n for n, _ in lengths[: args.top]]} tokens", flush=True)

    model, dtype = _load_qlora_model(config)
    model.train()
    budget = cap * args.headroom
    results, chosen = [], None
    for size in sizes:
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        peak, error = None, None
        try:
            for sample in longest:
                batch = collator([build_example(sample, size)])
                batch = {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in batch.items()}
                with torch.autocast("cuda", dtype=dtype):
                    loss = model(**batch).loss
                loss.backward()
                model.zero_grad(set_to_none=True)
                del batch, loss
            torch.cuda.synchronize()
            peak = torch.cuda.max_memory_reserved()
        except torch.OutOfMemoryError:
            error = "out of memory"
        model.zero_grad(set_to_none=True)
        gc.collect()
        torch.cuda.empty_cache()
        fits = peak is not None and peak <= budget
        results.append({"image_max_side": size, "peak_reserved_gb": peak and peak / 1024**3,
                        "fits": fits, "error": error})
        shown = f"{peak / 1024**3:.2f} GB" if peak else error
        print(f"image_max_side={size}: peak {shown} (budget {budget / 1024**3:.2f} GB) → "
              f"{'fits' if fits else 'too big'}", flush=True)
        if fits:
            chosen = size
            break
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    Path(args.out).write_text(json.dumps(
        {"chosen_image_max_side": chosen, "cap_gb": cap / 1024**3, "results": results,
         "longest_tokens": [n for n, _ in lengths[: args.top]]}, indent=2) + "\n")
    if chosen is None:
        raise SystemExit("No image size fits the VRAM limit — raise VRAM_LIMIT_GB in Settings.")
    print(f"chosen image_max_side = {chosen}")
    return 0


def cmd_smoke(args: argparse.Namespace) -> int:
    from training.qlora.data import load_training_split
    from training.qlora.train import projected_hours, run_training, total_steps

    cap_vram()
    patch_trainer_for_windows()
    out = Path(args.out)
    shutil.rmtree(out, ignore_errors=True)
    config = load_train_config(args)
    smoke = load_train_config(args, **{
        "run_name": "smoke", "data.train_limit": 8, "data.eval_samples": 2,
        "optimization.max_steps": 2, "optimization.gradient_accumulation_steps": 4,
        "checkpointing.save_steps": 1, "checkpointing.eval_steps": 2,
        "checkpointing.logging_steps": 1,
    })
    result = run_training(smoke, Path(args.data), out)
    meta = json.loads(Path(result.metadata_path).read_text())
    log = [json.loads(line) for line in (out / "train_log.jsonl").read_text().splitlines()]
    speed = next(e for e in reversed(log) if "train_samples_per_second" in e)
    seconds_per_sample = 1 / speed["train_samples_per_second"]
    print(f"trainable params {meta['trainable_parameters']:,} of {meta['total_parameters']:,}; "
          f"peak GPU memory {meta['peak_gpu_memory_gb']:.1f} GB; {seconds_per_sample:.1f} s/sample")
    _, train = load_training_split(Path(args.data), config.data.train_split)
    opt = config.optimization
    steps = total_steps(len(train), opt.epochs, opt.gradient_accumulation_steps, opt.max_steps)
    hours = projected_hours(seconds_per_sample, len(train), opt.epochs)
    print(f"Full run: {steps} optimizer steps, ≈ {hours:.1f} h of training "
          f"(checkpoint every {config.checkpointing.save_steps} steps; the smoke run's first "
          "steps are slower than steady state, so this is an upper estimate)")
    return 0


def cmd_train(args: argparse.Namespace) -> int:
    from training.qlora.train import run_training

    out = Path(args.out)
    finished = out / "training_run.json"
    if finished.exists() and (out / "adapter" / "adapter_config.json").exists():
        print("Already trained:", json.loads(finished.read_text())["completed_at"])
        return 0
    cap_vram()
    patch_trainer_for_windows()
    config = load_train_config(args)
    print(f"run {config.run_name}: image_max_side {config.data.image_max_side}, "
          f"{config.quantization.compute_dtype} compute → {out}", flush=True)
    started = time.time()
    result = run_training(config, Path(args.data), out)
    print(result)
    print(f"finished in {(time.time() - started) / 3600:.2f} h")
    return 0


def cmd_handback(args: argparse.Namespace) -> int:
    from training.qlora.handback import package_handback
    from training.qlora.train import environment

    manifest = package_handback(
        Path(args.out),
        training_dirs=[Path(p) for p in args.train],
        eval_dirs=[Path(p) for p in args.eval],
        extra_files=[Path(p) for p in args.extra],
        environment=environment(),
    )
    size = Path(args.out).stat().st_size / 1024**2
    print(f"Hand-back: {args.out} ({size:.1f} MB, {len(manifest['files'])} files)")
    if manifest["missing"]:
        print("Missing (re-run the section that produces them):", *manifest["missing"],
              sep="\n  ")
    return 0


def cmd_module(argv: list[str]) -> int:
    if not argv:
        raise SystemExit("module: name a python module to run")
    module, *rest = argv
    if "--backend" in rest and rest[rest.index("--backend") + 1 : rest.index("--backend") + 2] == ["hf"]:
        cap_vram()  # a model runs on the GPU: apply the memory limit
    sys.argv = [module, *rest]
    runpy.run_module(module, run_name="__main__", alter_sys=True)
    return 0


def main() -> int:
    os.chdir(REPO)
    if len(sys.argv) > 1 and sys.argv[1] == "module":
        return cmd_module(sys.argv[2:])
    parser = argparse.ArgumentParser(prog="vigraph_windows.py")
    commands = parser.add_subparsers(dest="command", required=True)
    commands.add_parser("gpu")
    for name in ("check", "probe", "smoke", "train"):
        sub = commands.add_parser(name)
        sub.add_argument("--data", required=True)
        sub.add_argument("--config", required=True)
        sub.add_argument("--overrides", default="{}")
        if name != "check":
            sub.add_argument("--out", required=True)
        if name == "probe":
            sub.add_argument("--sizes", default="1024,896,768,640")
            sub.add_argument("--top", type=int, default=3)
            sub.add_argument("--headroom", type=float, default=0.9)
    hand = commands.add_parser("handback")
    hand.add_argument("--out", required=True)
    hand.add_argument("--train", nargs="*", default=[])
    hand.add_argument("--eval", nargs="*", default=[])
    hand.add_argument("--extra", nargs="*", default=[])
    args = parser.parse_args()
    handler = {"gpu": cmd_gpu, "check": cmd_check, "probe": cmd_probe, "smoke": cmd_smoke,
               "train": cmd_train, "handback": cmd_handback}[args.command]
    return handler(args)


if __name__ == "__main__":
    raise SystemExit(main())
'''
DRIVER.write_text(DRIVER_SOURCE, encoding="utf-8", newline="\n")
print("python:", PY, sys.version.split()[0])
print("root:  ", ROOT, "| VRAM limit:", VRAM_LIMIT_GB, "GB | run:", RUN_NAME)

### 1.3 System check — GPU driver, Graphviz (portable download if missing), Tesseract, disk

No admin rights needed: if Graphviz is not installed, a portable copy is downloaded into
`E:\Asif\vigraph\tools\graphviz` (~100 MB, once).

In [ ]:
import zipfile

problems = []

# --- NVIDIA driver ------------------------------------------------------------------------
smi = shutil.which("nvidia-smi")
if smi:
    info = subprocess.run([smi, "--query-gpu=name,memory.total,driver_version",
                           "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    print("GPU:", info)
    try:
        major = int(info.split(",")[-1].strip().split(".")[0])
    except ValueError:
        major = 0
    if major < 570:
        print("  note: driver older than 570 — usually still fine; the GPU test (1.6) decides.")
else:
    problems.append("nvidia-smi not found — the NVIDIA driver is missing (ask IT to install it).")

# --- Graphviz: installed, or portable download --------------------------------------------
ENGINES = ("dot", "twopi", "circo", "neato")
if not all(shutil.which(e) for e in ENGINES):
    print("Graphviz not found — downloading the portable version (no install needed) …")
    target = TOOLS / "graphviz"
    for gv in GRAPHVIZ_VERSIONS:
        try:
            archive = download(GRAPHVIZ_URL.format(v=gv), TOOLS / f"graphviz-{gv}.zip")
            remove_tree(target)
            with zipfile.ZipFile(archive) as z:
                z.extractall(target)
            archive.unlink()
            break
        except Exception as exc:
            print(f"  Graphviz {gv}: download failed ({exc})")
    for bin_dir in sorted(target.glob("*/bin")) + sorted(target.glob("bin")):
        add_to_path(bin_dir)
    dot_exe = shutil.which("dot")
    if dot_exe:  # register the plugins (writes a config file next to dot.exe; no admin)
        subprocess.run([dot_exe, "-c"], capture_output=True)
if all(shutil.which(e) for e in ENGINES):
    print("graphviz:", shutil.which("dot"))
    print("         ", subprocess.run([shutil.which("dot"), "-V"], capture_output=True,
                                      text=True).stderr.strip())
    for engine in ENGINES:
        test = subprocess.run([shutil.which(engine), "-Tpng"], input=b"digraph{a->b; b->c}",
                              capture_output=True)
        if test.returncode or not test.stdout.startswith(b"\x89PNG"):
            subprocess.run([shutil.which("dot"), "-c"], capture_output=True)  # retry once
            test = subprocess.run([shutil.which(engine), "-Tpng"], input=b"digraph{a->b}",
                                  capture_output=True)
        if test.returncode or not test.stdout.startswith(b"\x89PNG"):
            problems.append(f"Graphviz '{engine}' cannot render PNG: "
                            + test.stderr.decode(errors="replace")[:300])
            break
    else:
        print("graphviz: dot, twopi, circo, neato render PNG — OK")
else:
    problems.append("Graphviz could not be downloaded. Download the 'win64 .zip' from "
                    "https://graphviz.org/download/ , unzip it into E:\\Asif\\vigraph\\tools\\graphviz, "
                    "and re-run section 1.")

# --- Tesseract (optional: only the OCR baseline uses it) ----------------------------------
USE_BASELINE = RUN_BASELINE and bool(shutil.which("tesseract"))
if RUN_BASELINE and not USE_BASELINE:
    print("tesseract: not found — the OCR baseline will be skipped (the VLM training and the "
          "zero-shot vs fine-tuned comparison are unaffected).")
elif USE_BASELINE:
    print("tesseract:", subprocess.run([shutil.which("tesseract"), "--version"],
                                       capture_output=True, text=True).stdout.splitlines()[0])

# --- Disk ---------------------------------------------------------------------------------
free_gb = shutil.disk_usage(ROOT).free / 1024**3
print(f"free space on {ROOT.drive}: {free_gb:.0f} GB")
if free_gb < 30:
    problems.append(f"Only {free_gb:.0f} GB free on {ROOT.drive} — about 30 GB is needed.")

fonts = Path(os.environ.get("WINDIR", r"C:\Windows")) / "Fonts"
if not any(fonts.glob("DejaVuSans*.ttf")):
    print("note: DejaVu fonts are not installed — Graphviz uses a fallback font. That is fine "
          "(images differ in pixels only; the ground truth is verified in section 2).")

if problems:
    raise RuntimeError("Fix these, then re-run section 1:\n  - " + "\n  - ".join(problems))
print("\nSystem check passed.")

### 1.4 Code — download the repo (or update it)

Uses Git if it happens to be installed; otherwise downloads the branch as a zip from GitHub.
Either way the files are byte-identical to Linux/Colab. It re-downloads only when the branch
has a new commit.

In [ ]:
import tempfile
import urllib.request
import zipfile

OWNER_REPO = REPO_URL.removeprefix("https://github.com/").removesuffix(".git")
git = shutil.which("git")

if git and (REPO / ".git").exists():
    run([git, "-C", REPO, "config", "core.autocrlf", "false"], log="setup")
    run([git, "-C", REPO, "fetch", "origin", BRANCH], log="setup")
    run([git, "-C", REPO, "checkout", BRANCH], log="setup")
    run([git, "-C", REPO, "pull", "--ff-only", "origin", BRANCH], log="setup")
    run([git, "-C", REPO, "log", "-1", "--oneline"])
else:
    stamp = REPO / ".vigraph_commit"
    latest = None
    try:
        request = urllib.request.Request(
            f"https://api.github.com/repos/{OWNER_REPO}/commits/{BRANCH}",
            headers={"Accept": "application/vnd.github.sha", "User-Agent": "vigraph-notebook"})
        latest = urllib.request.urlopen(request, timeout=30).read().decode().strip()
    except Exception as exc:
        print("could not check the latest commit:", exc)
    current = stamp.read_text(encoding="utf-8").strip() if stamp.exists() else None
    if current and (latest is None or latest == current):
        print(f"code up to date: {BRANCH} @ {current[:7]}")
    else:
        archive = download(f"https://github.com/{OWNER_REPO}/archive/refs/heads/{BRANCH}.zip",
                           ROOT / "tools" / "vigraph-code.zip")
        with tempfile.TemporaryDirectory(dir=ROOT) as tmp:
            with zipfile.ZipFile(archive) as z:
                z.extractall(tmp)
            (top,) = [p for p in Path(tmp).iterdir() if p.is_dir()]
            remove_tree(REPO)
            shutil.move(str(top), str(REPO))
        archive.unlink()
        if latest:
            stamp.write_text(latest + "\n", encoding="utf-8")
        print(f"code downloaded: {BRANCH} @ {(latest or 'unknown')[:7]} → {REPO}")
for required in ("requirements-train.txt", "training/configs/qlora_t4.yaml",
                 "data/splits/synthetic-v1.manifest.json"):
    if not (REPO / required).exists():
        raise RuntimeError(f"{REPO / required} is missing — delete {REPO} and re-run this cell.")
print("code OK")

### 1.5 Libraries

PyTorch with CUDA is a 2.8 GB file from the PyTorch index (PyPI only has CPU builds for
Windows). pip cannot survive connection drops on a file that big, so this cell downloads it
itself in 64 MB pieces — a dropped connection just retries the missing piece — checks the
official sha256, and then pip installs it from `E:\Asif\vigraph\tools\wheels`. The installed torch
is pinned as a constraint so no later package can swap it for a CPU build.
Re-running is quick: finished downloads and installs are skipped.

In [ ]:
import hashlib, re, urllib.parse, urllib.request
from importlib.metadata import version as installed, PackageNotFoundError

WHEELS = ROOT / "tools" / "wheels"
UA = {"User-Agent": "vigraph-notebook"}


def pip(*args):
    run([PY, "-m", "pip", "install", "--disable-pip-version-check", *args], log="setup")


def find_wheel(package, version):
    # (url, filename, sha256) of the CUDA wheel for this Python on Windows
    tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    cuda = TORCH_INDEX.rstrip("/").rsplit("/", 1)[-1]                  # "cu128"
    wanted = f"{package}-{version}+{cuda}-{tag}-{tag}-win_amd64.whl"
    page_url = f"{TORCH_INDEX.rstrip('/')}/{package}/"
    for attempt in range(1, 6):
        try:
            with urllib.request.urlopen(urllib.request.Request(page_url, headers=UA),
                                        timeout=60) as response:
                page = response.read().decode("utf-8", "replace")
            break
        except Exception as exc:
            if attempt == 5:
                raise RuntimeError(f"cannot read {page_url}: {exc}") from exc
            time.sleep(5 * attempt)
    for href in re.findall(r'href="([^"]+)"', page):
        link, _, fragment = href.partition("#")
        if urllib.parse.unquote(link.rsplit("/", 1)[-1]) == wanted:
            sha = fragment.split("=", 1)[1] if fragment.startswith("sha256=") else None
            return urllib.parse.urljoin(page_url, link), wanted, sha
    raise RuntimeError(f"{wanted} is not on {page_url}")


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while block := handle.read(1 << 24):
            digest.update(block)
    return digest.hexdigest()


def fetch(url, dest, sha256=None, piece=64 << 20, max_failures=40):
    # Resumable download in byte ranges; survives dropped connections
    if dest.exists() and (sha256 is None or sha256_of(dest) == sha256):
        print(f"{dest.name}: already downloaded")
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_name(dest.name + ".part")
    head = urllib.request.Request(url, headers=UA, method="HEAD")
    with urllib.request.urlopen(head, timeout=60) as response:
        total = int(response.headers["Content-Length"])
    done = part.stat().st_size if part.exists() else 0
    if done > total:
        part.unlink()
        done = 0
    failures, started, start_bytes = 0, time.time(), done
    while done < total:
        end = min(done + piece, total) - 1
        request = urllib.request.Request(url, headers={**UA, "Range": f"bytes={done}-{end}"})
        try:
            with urllib.request.urlopen(request, timeout=60) as response, open(part, "ab") as out:
                if response.status != 206:
                    raise OSError(f"server ignored the byte range (HTTP {response.status})")
                while block := response.read(1 << 20):
                    out.write(block)
                    done += len(block)
            if done <= end:
                raise OSError("connection closed early")
            failures = 0
        except Exception as exc:
            failures += 1
            if failures > max_failures:
                raise RuntimeError(f"download kept failing ({exc}); re-run this cell to resume")
            wait = min(60, 5 * failures)
            print(f"\n  connection problem ({exc}); retrying in {wait}s "
                  f"(attempt {failures}/{max_failures}) …")
            time.sleep(wait)
        speed = (done - start_bytes) / max(time.time() - started, 1e-6) / 1024**2
        print(f"\r  {dest.name}: {done / 1024**3:.2f} / {total / 1024**3:.2f} GB "
              f"({speed:.0f} MB/s)   ", end="", flush=True)
    print()
    if sha256:
        print("  checking sha256 …")
        if sha256_of(part) != sha256:
            part.unlink()
            raise RuntimeError(f"{dest.name} is corrupted (sha256 mismatch) — re-run this cell")
    part.replace(dest)
    return dest


pip("--upgrade", "pip")

try:
    have_torch = installed("torch")
except PackageNotFoundError:
    have_torch = None
if have_torch and "+cu" in have_torch:
    print("torch already installed:", have_torch)
else:
    wheels = []
    for package in ("torch", "torchvision"):
        url, name, sha = find_wheel(package, PINS[package])
        wheels.append(fetch(url, WHEELS / name, sha))
    pip(*wheels)       # their small dependencies come from PyPI

constraints = OUT / "constraints.txt"
lines = [f"torch=={installed('torch')}", f"torchvision=={installed('torchvision')}"]
lines += [f"{name}=={v}" for name, v in PINS.items() if name not in ("torch", "torchvision")]
constraints.write_text("\n".join(lines) + "\n", encoding="utf-8")
pip("-r", REPO / "requirements-train.txt", "matplotlib", "-c", constraints)

try:   # an old torchao breaks adapter loading (seen in Colab); it is not needed here
    installed("torchao")
    run([PY, "-m", "pip", "uninstall", "-y", "torchao"], log="setup")
except PackageNotFoundError:
    pass

check = subprocess.run([PY, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
                       capture_output=True, text=True, env=ENV)
print("\ntorch:", check.stdout.strip() or check.stderr.strip())
if "True" not in check.stdout:
    raise RuntimeError("torch cannot see the GPU (CPU-only build or old driver). "
                       "Run:  .venv\\Scripts\\python.exe -m pip uninstall -y torch torchvision  "
                       "and re-run this cell.")
for name in ("transformers", "peft", "bitsandbytes", "accelerate", "numpy", "pillow"):
    print(f"{name:13s}", installed(name))

### 1.6 GPU test — memory cap, bf16, bitsandbytes 4-bit + paged 8-bit AdamW

Runs a tiny 4-bit layer and one optimizer step on the GPU, so library problems show up now
rather than hours into training.

In [ ]:
driver("gpu", log="setup")

## 2. Dataset — check the one the main notebook built

Verifies the ground truth of `data\synthetic-v1` against the repo (seconds). If the dataset is
missing it is built again (deterministic, ~10 min). The evaluation runner also re-hashes every
image and ground-truth file before each run.

In [ ]:
manifest_path = DATA_DIR / "manifest.json"
if not manifest_path.exists():
    build = ["module", "data.generator", "build", "--out", DATA_DIR, "--workers", DATA_WORKERS]
    if DATA_DIR.exists() and any(DATA_DIR.iterdir()):
        build.append("--overwrite")          # an interrupted build: start it again
    driver(*build, log="dataset")
driver("module", "data.generator", "verify", DATA_DIR,
       "--reference", REPO / "data" / "splits" / "synthetic-v1.manifest.json", log="dataset")
manifest = read_json(manifest_path)
print("Graphviz:", manifest["graphviz_version"], "| dataset hash", manifest["dataset_hash"][:12])

## A. Greedy decoding — all 500 test samples

The headline numbers on the whole held-out test split, with the same settings as the latest
runs: 896 px, 2048 tokens, greedy, runaway guard on. The fine-tuned model goes first.

| Cell | Run | Time (rough) |
| --- | --- | --- |
| A.2 | Fine-tuned (QLoRA) | ~5–6 h |
| A.3 | Zero-shot | ~6–8 h |
| A.4 | OCR baseline (CPU, needs Tesseract) | ~30 min |
| A.5 | Paired comparisons + results table | 1 min |
| A.6 | Hand-back zip — **send it before starting B** | 1 min |

In [ ]:
# A.1 Final-evaluation helpers (sections A and B use them; re-run after a restart)
import gzip
import yaml

TEST_SIZE = 500
model_cfg = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))["model"]
MODEL_SLUG = model_cfg["id"].split("/")[-1].lower()
ADAPTER = RUN_DIR / "adapter"
VLM = ["--backend", "hf", "--model-id", model_cfg["id"], "--dtype", COMPUTE_DTYPE,
       "--device-map", "cuda:0", "--max-new-tokens", MAX_NEW_TOKENS]
if model_cfg.get("revision"):
    VLM += ["--revision", model_cfg["revision"]]
if RUNAWAY_GUARD:
    VLM.append("--runaway-guard")
GREEDY = {"qlora": f"{RUN_NAME}-final-greedy",
          "zeroshot": f"{MODEL_SLUG}-zeroshot-final-greedy",
          "baseline": "baseline-v1-final"}
COMPARE_A = EVAL_DIR / "comparisons-final-greedy"
COMPARE_B = EVAL_DIR / "comparisons-final-seeds"


def seed_name(model, seed):
    stem = RUN_NAME if model == "qlora" else f"{MODEL_SLUG}-zeroshot"
    return f"{stem}-final-t{TEMPERATURE:g}-s{seed}"


def current_gpu():
    smi = shutil.which("nvidia-smi")
    if not smi:
        return None
    names = subprocess.run([smi, "--query-gpu=name", "--format=csv,noheader"],
                           capture_output=True, text=True).stdout.strip().splitlines()
    return names[0].strip() if names else None


def finished(path, expected=TEST_SIZE):
    if not ((path / "summary.json").exists() and (path / "run.json").exists()):
        return False
    info = read_json(path / "run.json")
    return bool(info.get("completed_at")) and info.get("samples_done") == expected


def run_gpu(path):
    return read_json(path / "run.json").get("environment", {}).get("gpu") if (
        path / "run.json").exists() else None


def evaluate(name, *args, limit=None):
    # Run (or resume) one evaluation on the test split; skip it when finished
    out = EVAL_DIR / name
    if finished(out, limit or TEST_SIZE):
        print(f"{name}: done")
        return out
    command = ["module", "evaluation", "run", "--dataset", DATA_DIR, "--split", "test",
               "--out", out, "--image-max-side", IMAGE_MAX_SIDE, *args]
    if limit:
        command += ["--limit", limit]
    started = time.time()
    driver(*command, log=f"eval-{name}")
    info = read_json(out / "run.json")
    print(f"\n{name}: {info['samples_done']} samples ({(time.time() - started) / 3600:.2f} h "
          "in this session)")
    if info.get("errors"):
        print(f"Note: {info['errors']} sample(s) raised an error; they count as failures.")
    return out


def predictions(path):
    raw = path / "predictions.jsonl"
    text = raw.read_text(encoding="utf-8") if raw.exists() else gzip.decompress(
        (path / "predictions.jsonl.gz").read_bytes()).decode("utf-8")
    return [json.loads(line) for line in text.splitlines() if line.strip()]


def by_level(path):
    # {level: (samples, cut off at the token limit, stopped by the guard, failed)}
    counts = {}
    for result in predictions(path):
        attempts = (result["prediction"].get("analysis") or {}).get("extraction", {}).get(
            "attempts", [])
        reasons = [a["output"].get("finish_reason") for a in attempts]
        level = result["attributes"]["level"]
        n, cut, stopped, failed = counts.get(level, (0, 0, 0, 0))
        counts[level] = (n + 1, cut + ("length" in reasons), stopped + ("runaway" in reasons),
                         failed + (not result["scores"]["has_graph"]))
    return counts


KEYS = ("node_f1", "edge_f1", "edge_strict_f1", "graph_similarity", "label_accuracy",
        "qa_accuracy", "diagram_type_accuracy", "valid_first_attempt", "valid_post_repair")


def results_table(runs, expected=TEST_SIZE):
    print(" | ".join(("run", "n", "node F1", "edge F1", "strict", "graph sim", "labels", "QA",
                      "type", "valid@1", "valid", "s/sample", "GPU")))
    for name, path in runs.items():
        if not finished(path, expected):
            print(f"{name} | not finished")
            continue
        summary = read_json(path / "summary.json")
        values = [summary["macro"].get(k, {}).get("mean") for k in KEYS]
        latency = summary.get("latency_ms", {}).get("mean")
        print(" | ".join([name, str(summary["samples"]),
                          *("–" if v is None else f"{v:.3f}" for v in values),
                          "–" if latency is None else f"{latency / 1000:.0f}",
                          str(run_gpu(path))]))
    print("\nper level: cut off / stopped by the guard / failed (of n), and graph similarity:")
    for name, path in runs.items():
        if finished(path, expected):
            levels = read_json(path / "summary.json")["breakdowns"]["level"]
            cells = [f"L{lv}: {c}/{s}/{f} of {n}, {levels[str(lv)]['graph_similarity']:.3f}"
                     for lv, (n, c, s, f) in sorted(by_level(path).items())]
            print(f"  {name:22s}", "  ".join(cells))


GPU = current_gpu()
if not (ADAPTER / "adapter_config.json").exists():
    raise RuntimeError(f"No trained adapter in {ADAPTER} — check ROOT and RUN_NAME in 1.1.")
print(f"GPU {GPU} | {IMAGE_MAX_SIDE} px | max_new_tokens {MAX_NEW_TOKENS} | "
      f"runaway guard {'on' if RUNAWAY_GUARD else 'off'} | adapter {ADAPTER}")
for name in GREEDY.values():
    path = EVAL_DIR / name
    state = (f"done on {run_gpu(path)}" if finished(path) else
             "started — its cell resumes it" if (path / "predictions.jsonl").exists() else "to run")
    print(f"  {name}: {state}")

In [ ]:
# A.2 Fine-tuned model, greedy, all 500 (the most important run — resumable)
qlora_greedy = evaluate(GREEDY["qlora"], *VLM, "--adapter", ADAPTER)

In [ ]:
# A.3 Zero-shot model, greedy, all 500 (resumable)
zeroshot_greedy = evaluate(GREEDY["zeroshot"], *VLM)

In [ ]:
# A.4 OCR baseline, all 500 (CPU; resumable)
if USE_BASELINE:
    baseline = evaluate(GREEDY["baseline"], "--backend", "baseline")
else:
    print("OCR baseline skipped: Tesseract was not found (section 1.3). The VLM results are "
          "unaffected; the repo has a 500-sample baseline run from another data build.")

In [ ]:
# A.5 Paired comparisons (§20.11) and results — paste this output back with the zip
runs_a = {model: EVAL_DIR / name for model, name in GREEDY.items()}
COMPARE_A.mkdir(parents=True, exist_ok=True)
for title, (a, b) in {"qlora-vs-zeroshot": ("zeroshot", "qlora"),
                      "qlora-vs-baseline": ("baseline", "qlora"),
                      "zeroshot-vs-baseline": ("baseline", "zeroshot")}.items():
    if finished(runs_a[a]) and finished(runs_a[b]):
        driver("module", "evaluation", "compare", "--a", runs_a[a], "--b", runs_a[b],
               "--out", COMPARE_A / f"{title}.md", log="compare-final")
    else:
        print(f"skipped {title}: not both finished")
print()
results_table(runs_a)

In [ ]:
# A.6 Hand-back for section A — send this zip before starting section B
handback_a = OUT / "handback" / f"vigraph-handback-{RUN_NAME}-final-greedy.zip"
done_a = [path for path in runs_a.values() if finished(path)]
extra_a = sorted(COMPARE_A.glob("*")) + [DATA_DIR / "manifest.json"]
driver("handback", "--out", handback_a, "--eval", *done_a, "--extra", *extra_a, log="handback")
print("\nGive back:", handback_a)

## B. Sampling — 3 seeds per model

Run-to-run variance (§20.10). Greedy decoding gives the same answer every time, so the variance
comes from sampling: each model runs once per seed with `TEMPERATURE` and `TOP_P` (settings
1.1); everything else is as in section A. The runs alternate fine-tuned / zero-shot seed by
seed, so stopping early still leaves complete pairs.

With `SEED_LIMIT = None` every run covers all 500 samples (~35–45 h in total). To finish in
~9 h, set `SEED_LIMIT = 112` in 1.1 (the stratified subset used for every earlier comparison)
and re-run 1.1 and A.1 — before B.2 starts, not halfway.

In [ ]:
# B.1 The sampled runs (needs A.1)
SAMPLING = ["--temperature", TEMPERATURE, "--top-p", TOP_P]
SEED_RUNS = [(model, seed) for seed in SEEDS for model in ("qlora", "zeroshot")]
SEED_EXPECTED = SEED_LIMIT or TEST_SIZE
print(f"temperature {TEMPERATURE}, top-p {TOP_P}, {SEED_EXPECTED} samples per run, "
      f"seeds {SEEDS} | GPU {GPU}")
for model, seed in SEED_RUNS:
    path = EVAL_DIR / seed_name(model, seed)
    state = (f"done on {run_gpu(path)}" if finished(path, SEED_EXPECTED) else
             "started — B.2 resumes it" if (path / "predictions.jsonl").exists() else "to run")
    print(f"  {seed_name(model, seed)}: {state}")

In [ ]:
# B.2 Run (or resume) every sampled run in order — after a stop, run 1, 2, A.1, B.1, then this
for model, seed in SEED_RUNS:
    args = [*VLM, *SAMPLING, "--seed", seed]
    if model == "qlora":
        args += ["--adapter", ADAPTER]
    evaluate(seed_name(model, seed), *args, limit=SEED_LIMIT)

In [ ]:
# B.3 Seed statistics and comparisons — paste this output back with the zip
groups = {model: [EVAL_DIR / seed_name(model, seed) for seed in SEEDS
                  if finished(EVAL_DIR / seed_name(model, seed), SEED_EXPECTED)]
          for model in ("qlora", "zeroshot")}
COMPARE_B.mkdir(parents=True, exist_ok=True)
for model, runs in groups.items():
    if len(runs) >= 2:
        driver("module", "evaluation", "seeds", *runs, "--out", COMPARE_B / f"{model}-seeds.md",
               log="seeds-final")
    else:
        print(f"{model}: {len(runs)} finished seed run(s) — at least 2 are needed for a spread")
if groups["qlora"] and groups["zeroshot"]:
    driver("module", "evaluation", "compare", "--a", *groups["zeroshot"], "--b", *groups["qlora"],
           "--out", COMPARE_B / "qlora-vs-zeroshot-seeds.md", log="compare-final")
if SEED_EXPECTED == TEST_SIZE:  # same samples as section A: sampling vs greedy
    for model in ("qlora", "zeroshot"):
        greedy = EVAL_DIR / GREEDY[model]
        if groups[model] and finished(greedy):
            driver("module", "evaluation", "compare", "--a", greedy, "--b", *groups[model],
                   "--out", COMPARE_B / f"{model}-sampled-vs-greedy.md", log="compare-final")
print()
results_table({seed_name(m, s): EVAL_DIR / seed_name(m, s) for m, s in SEED_RUNS}, SEED_EXPECTED)

In [ ]:
# B.4 Hand-back for section B
handback_b = OUT / "handback" / f"vigraph-handback-{RUN_NAME}-final-seeds.zip"
done_b = [path for runs in groups.values() for path in runs]
driver("handback", "--out", handback_b, "--eval", *done_b,
       "--extra", *sorted(COMPARE_B.glob("*")), log="handback")
print("\nGive back:", handback_b)

### Files to give back

1. **After section A:** `E:\Asif\vigraph\output\handback\vigraph-handback-qwen3vl-2b-qlora-a6000-v1-final-greedy.zip`
   (~2 MB: the three runs with every raw output, the comparisons, the data manifest,
   `MANIFEST.json`) and the output of cell **A.5**.
2. **After section B:** `E:\Asif\vigraph\output\handback\vigraph-handback-qwen3vl-2b-qlora-a6000-v1-final-seeds.zip`
   and the output of cell **B.3**.

If a cell fails, send its output, or the matching log from `E:\Asif\vigraph\output\logs\`.